# 🎯 [K15_M01] 시설 예약·점검 정보의 검색 문제 파악하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K15/k15_m01_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군수시설·자산관리 (Level 1) / K15 군 시설 정보 검색 |
| **미션** | 미션 01. 시설 예약·점검 정보의 검색 문제 파악하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 조건 제어 프롬프트 엔지니어링 및 팩트체크 검증 (HITL) |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Colab 환경 설정] 데이터 분석 및 프롬프트 테스트 환경 준비
# ============================================================================
!pip install -q pandas numpy tabulate

import pandas as pd
import numpy as np
import json
import re

# 가상 한빛대대 데이터셋 인메모리 로드 (Colab 환경 독립 실행 보장)
schedule_raw = [
    {"facility": "대대 강당", "day": "월", "09-11": "1중대 교육", "11-13": "1중대 교육", "14-16": "훈련 평가", "16-18": "빈", "19-21": "빈"},
    {"facility": "대대 강당", "day": "화", "09-11": "2중대 교육", "11-13": "빈", "14-16": "3중대 교육", "16-18": "빈", "19-21": "합창 동아리"},
    {"facility": "대대 강당", "day": "수", "09-11": "3중대 교육", "11-13": "3중대 교육", "14-16": "빈", "16-18": "정훈 방송", "19-21": "빈"},
    {"facility": "대대 강당", "day": "목", "09-11": "1중대 안전교육", "11-13": "빈", "14-16": "2중대 교육", "16-18": "2중대 교육", "19-21": "빈"},
    {"facility": "대대 강당", "day": "금", "09-11": "월례 조회", "11-13": "점검(음향)", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "월", "09-11": "분대장 회의", "11-13": "빈", "14-16": "분대장 회의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "화", "09-11": "빈", "11-13": "인사과 상담", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "수", "09-11": "빈", "11-13": "빈", "14-16": "1중대 토의", "16-18": "1중대 토의", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "목", "09-11": "2중대 토의", "11-13": "2중대 토의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "금", "09-11": "빈", "11-13": "빈", "14-16": "3중대 토의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "토", "09-11": "독서 모임", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "월", "09-11": "빈", "11-13": "교육 준비", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "화", "09-11": "점검(빔프로젝터)", "11-13": "점검(빔프로젝터)", "14-16": "점검(빔프로젝터)", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "수", "09-11": "3중대 토의", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "목", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "AI 활용 스터디", "19-21": "AI 활용 스터디"},
    {"facility": "세미나실 B", "day": "금", "09-11": "대대 간부 회의", "11-13": "대대 간부 회의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "토", "09-11": "빈", "11-13": "자격증 스터디", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "월", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "화", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "수", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "소독(이용 불가)", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "목", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "3중대 단체 운동"},
    {"facility": "체력단련실(단체)", "day": "금", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "대대 체육 연습", "16-18": "대대 체육 연습", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"}
]

maintenance_raw = [
    {"id": "H01", "date": "08-03", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H02", "date": "08-10", "facility": "대대 강당", "type": "정기점검", "content": "음향 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H03", "date": "08-17", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H04", "date": "08-24", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H05", "date": "08-31", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H06", "date": "09-02", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 전원 꺼짐", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H07", "date": "09-07", "facility": "체력단련실", "type": "정기점검", "content": "운동 기구 안전 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H08", "date": "09-09", "facility": "사이버지식정보방", "type": "고장신고", "content": "PC 3대 부팅 불가", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H09", "date": "09-14", "facility": "대대 강당", "type": "고장신고", "content": "출입문 잠금장치 불량", "dept": "군수과", "status": "완료", "days": 5},
    {"id": "H10", "date": "09-16", "facility": "세미나실 A", "type": "정기점검", "content": "환기 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H11", "date": "09-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 6},
    {"id": "H12", "date": "09-23", "facility": "세미나실 B", "type": "정기점검", "content": "영상 장비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H13", "date": "09-28", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 3},
    {"id": "H14", "date": "10-01", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H15", "date": "10-05", "facility": "사이버지식정보방", "type": "정기점검", "content": "PC·네트워크 점검", "dept": "정보통신실", "status": "완료", "days": 1},
    {"id": "H16", "date": "10-08", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H17", "date": "10-13", "facility": "체력단련실", "type": "고장신고", "content": "샤워실 배수 불량", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H18", "date": "10-15", "facility": "대대 강당", "type": "정기점검", "content": "조명·음향 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H19", "date": "10-19", "facility": "세미나실 B", "type": "고장신고", "content": "에어컨 냉기 약함", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H20", "date": "10-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H21", "date": "10-22", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "접수", "days": np.nan},
    {"id": "H22", "date": "10-23", "facility": "세미나실 A", "type": "정기점검", "content": "난방기 점검", "dept": "군수과", "status": "완료", "days": 1}
]

df_schedule = pd.DataFrame(schedule_raw)
df_maintenance = pd.DataFrame(maintenance_raw)

print("✅ [K15] 가상 한빛대대 실습 환경 및 검산용 데이터프레임 로드 완료!")
print(f"- 1주 시설 예약 현황표: {len(df_schedule)}행 (4개 시설 × 7일)")
print(f"- 시설 점검 및 고장 이력: {len(df_maintenance)}건 (정기점검 7건, 고장신고 15건)")


---

## 🎯 미션 01. 시설 예약·점검 정보의 검색 문제 파악하기
**소요시간**: 약 1.5시간 | **난이도**: 초급 (Level 1) | **유형**: AI 활용형

### 📋 미션 개요 & 핵심 학습 목표
**[미션 개요]**
<p>부대 시설 예약 현황과 설비 점검 이력을 여러 문서에서 수작업으로 대조하며 발생하는 정보 확인 지연과 행정 비효율을 분석합니다. 실습용 질문 카드 20개를 예약 가능 여부, 점검 이력, 자료 밖 질문으로 분류하고, 기존 검색 방식에 소요되는 시간과 주요 불편 요소를 구체화합니다. 이를 통해 시설 정보 검색을 지원할 AI 도우미의 필요성을 도출하고 문제 해결의 기준을 정립합니다.</p>

**[학습 목표 및 체득 역량]**
<p>실습자료의 질문 카드 20개를 예약 가능 여부·점검 이력·자료 밖 질문으로 분류하고, 기존 수작업 검색 방식의 소요 시간과 불편 요소를 도출합니다. <strong>이를 통해 실무 현장의 정보 탐색 병목 현상을 진단하고, AI 도입이 필요한 작업 영역을 정확히 식별·정의하는 문제 분석 역량을 익힙니다.</strong></p>

### 🛠️ 세부 실습 활동 (GuideSteps)
#### 🔹 Step 1: 질문 카드 20개를 예약·점검·자료 밖으로 분류한다.
<h4>이 단계의 역할</h4><p>실습 데이터에 포함된 질문 카드 20개를 분석하여 질문의 유형(예약 현황, 점검·고장 이력, 자료 밖 질문)을 파악하고 분류표를 작성합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>질문 카드 20개의 질의 내용(시설, 요일, 시간대, 설비 고장 등)을 꼼꼼히 읽습니다.</p></li>
<li><p>각 카드를 '예약 가능 여부(1~7번)', '점검 및 고장 이력(8~15번)', '자료 밖 질문(16~19번)', '예약 대행 요청(20번)'으로 분류합니다.</p></li>
<li><p>표에 정답이 명시되어 있는 질문과 근거 데이터가 없어 답할 수 없는 질문의 차이점을 도출합니다.</p></li>
</ol>
<h4>💡 AI 프롬프트 질의 예시</h4><blockquote><p><strong>[질문 유형 분석 질의]:</strong> "부대 행정 실무에서 장병들이 공용 시설(강당, 세미나실, 체육시설)에 대해 자주 묻는 질문 20가지를 분석하려 해. '가용 일정 문의', '설비 상태/고장 이력 문의', '자료 밖 예외 질문'으로 나눌 때 효과적인 분류 기준과 판별 요령을 알려줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>분류 완전성:</strong> 20개 질문 카드 전체가 누락 없이 분류표에 매핑되었는가?</li>
<li><strong>자료 밖 질문 식별:</strong> 질문 16~19번(세미나실 C, 수용인원, 차주 일정, 승인 담당자)과 20번(예약 대행)이 자료 밖 질문으로 정확히 판정되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>질문 20개 분류 완료 및 자료 밖 질문 4개(16~19)와 대행 불가 질문(20) 식별</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>질문 카드 20개 유형 분류표</p></li></ul>

#### 🔹 Step 2: 지금의 검색 방식과 걸리는 시간을 적는다.
<h4>이 단계의 역할</h4><p>현재 부대에서 시설 예약 및 점검 현황을 확인할 때 사용하는 수작업 방식(게시판 표 확인, 유선 문의)과 소요 시간을 측정·정리합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>행정반 게시판에 부착된 예약표를 직접 찾아가 확인하거나 담당자에게 내선 전화로 문의할 때 걸리는 단계별 소요 시간을 기록합니다.</p></li>
<li><p>질문 1건당 평균 확인 소요 시간(이동, 대조, 담당관 부재 대기 등)을 산출합니다.</p></li>
<li><p>근무 교대나 훈련 기간 중 즉시 답변을 얻기 어려운 지연 요인을 파악합니다.</p></li>
</ol>
<h4>💡 AI 프롬프트 분석 예시</h4><blockquote><p><strong>[업무 소요 시간 분석 질의]:</strong> "부대 내에서 수작업으로 게시판 종이 표를 확인하거나 행정과에 유선으로 시설 가능 여부를 문의할 때 발생하는 평균 업무 체류 시간과 병목 요소 3가지를 정리해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>시간 측정 객관성:</strong> 실제 부대 환경을 가정한 이동 시간, 서류 열람 시간, 전화 대기 시간이 현실적으로 반영되었는가?</li>
<li><strong>병목 지점 도출:</strong> 단순 조회에 비해 점검 이력이나 고장 상태 파악 시 시간이 2배 이상 걸리는 이유를 규명했는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>기존 검색 방식(게시판 열람, 전화 문의)별 건당 소요 시간 측정치</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>현재 검색 방식 및 소요 시간 분석표</p></li></ul>

#### 🔹 Step 3: 불편한 점을 정리한다.
<h4>이 단계의 역할</h4><p>수작업 확인 과정에서 발생하는 주요 불편 요소와 정보 누락·오류 위험을 구체화합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>동일 시간대에 중복 예약이 발생하거나 점검 중인 시설을 빈 시설로 오인했던 과거 경험/가상 사례를 정리합니다.</p></li>
<li><p>시설 담당관의 잦은 자리 비움, 서류 수기 갱신 지연으로 인한 불편 사항을 항목화합니다.</p></li>
<li><p>AI 기반 검색 도우미 도입 시 해결할 수 있는 핵심 개선 과제 3가지를 도출합니다.</p></li>
</ol>
<h4>💡 AI 협업 설계 질의 예시</h4><blockquote><p><strong>[불편 요인 체계화 질의]:</strong> "종이 예약 현황표와 설비 수기 장부를 운용하는 환경에서 실무자가 겪는 가장 치명적인 불편 요소(중복 예약, 점검 미인지, 환각 답변 등)를 해결하기 위한 AI 도우미의 필수 요구사항을 정리해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>문제 구체성:</strong> 단순 '불편하다'가 아닌 '점검 중인 줄 모르고 예약하여 행사 당일 차질 발생' 등 실무적 파급 효과가 명시되었는가?</li>
<li><strong>도입 타당성:</strong> AI가 검색을 지원할 때 절감되는 행정 비용이 명확한가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>시설 정보 확인 시 불편 요소 3대 항목 및 개선 기대 효과 정리</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>불편한 점 정리서 및 AI 도입 필요성 보고 메모</p></li></ul>

#### 🔹 Step 4: 심화 도전 과제 및 최적화
<h4>이 단계의 역할</h4><p>기본 실습 완료 후 실제 겪었던 시설 조회 곤란 경험을 가상 질문 카드로 추가하여 분석을 심화합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>부대 생활 중 시설 이용이나 설비 고장으로 곤란했던 실제(또는 유사) 경험을 회상합니다.</p></li>
<li><p>해당 경험을 질문 카드 서식에 맞추어 1건 신규 제작합니다(질문 21번).</p></li>
<li><p>새로 만든 질문이 '예약', '점검', '자료 밖' 중 어디에 해당하는지 자가 진단합니다.</p></li>
</ol>
<h4>💡 AI 프롬프트 시뮬레이션 예시</h4><blockquote><p><strong>[신규 질문 생성 질의]:</strong> "부대 생활관이나 부속 시설을 이용할 때 장병들이 당황하기 쉬운 예외 상황(예: 갑작스러운 방송 장비 고장, 야간 체력단련실 소독 일정)을 반영한 실무 질문 1개를 만들어 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>질문 현실성:</strong> 부대 실무 현장에서 충분히 일어날 수 있는 조건인가?</li>
<li><strong>유형 판별 정확도:</strong> 신규 질문의 속성(예약/점검/자료밖)이 타당하게 정의되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>자가 경험 기반 추가 질문 카드 1건 제작 및 유형 분류</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>심화 도전 질문 카드 및 분류 메모</p></li></ul>

### 📑 실습 작성 서식 및 체크리스트 (Sample Template)

## 1. 질문 카드 20개 분류표
| 카드 번호 | 질문 내용 | 분류 (예약 가능 여부 / 점검 이력 / 자료 밖 / 대행 불가) | 자료에 답이 있는가? (O / X) |
| :---: | :--- | :---: | :---: |
| 1 | 수요일 저녁에 빈 강당 있어요? | 예약 가능 여부 | O |
| 2 | 목요일 오후에 세미나실 A, B 중 빈 곳은? | 예약 가능 여부 | O |
| 3 | 금요일 오전에 강당 쓸 수 있나요? | 예약 가능 여부 | O |
| 4 | 토요일에 하루 종일 비는 시설이 있나요? | 예약 가능 여부 | O |
| 5 | 월요일 오후 세미나실 A와 B가 둘 다 비는 시간은? | 예약 가능 여부 | O |
| 6 | 2중대가 이번 주 강당을 쓰는 시간은? | 예약 가능 여부 | O |
| 7 | 수요일에 체력단련실을 단체로 쓸 수 있는 시간은? | 예약 가능 여부 | O |
| 8 | 강당의 점검·고장 이력을 정리해 줘 | 점검·고장 이력 | O |
| 9 | 세미나실 B 빔프로젝터는 몇 번 고장 났어요? | 점검·고장 이력 | O |
| 10 | 가장 고장 신고가 잦은 시설은 어디예요? | 점검·고장 이력 | O |
| 11 | 아직 수리가 끝나지 않은 건은? | 점검·고장 이력 | O |
| 12 | 고장 신고 처리에 평균 며칠 걸렸어요? | 점검·고장 이력 | O |
| 13 | 강당 무선 마이크 고장 날짜를 알려 줘 | 점검·고장 이력 | O |
| 14 | 정기점검을 가장 오래 안 한 시설은? | 점검·고장 이력 | O |
| 15 | 반복해서 고장 나는 설비 목록을 알려 줘 | 점검·고장 이력 | O |
| 16 | 세미나실 C는 비어 있어요? | 자료 밖 질문 | X |
| 17 | 강당 최대 수용 인원이 몇 명이에요? | 자료 밖 질문 | X |
| 18 | 다음 주 월요일 강당 예약 현황은? | 자료 밖 질문 | X |
| 19 | 강당 예약 승인해 주는 담당자 이름은? | 자료 밖 질문 | X |
| 20 | 빈 강당 있으면 지금 예약해 줘 | 대행 불가 요청 | X |

## 2. 현재 검색 방식 및 소요 시간 분석표
| 확인 방식 | 수행 절차 | 건당 평균 소요 시간 | 주요 지연 요인 |
| :--- | :--- | :---: | :--- |
| 게시판 종이 표 확인 | 행정반 이동 -> 게시판 열람 -> 대조 | 약 15분 | 표 갱신 지연, 이동 시간 소요, 수기 대조 착오 위험 |
| 담당자 내선 문의 | 행정과/군수과 전화 -> 확인 요청 -> 대기 | 약 20분 | 담당관 부재(훈련, 순찰, 회의) 시 지연, 반복 문의로 인한 피로도 |

## 3. 현장 불편 요소 및 AI 검색 도우미 도입 필요성
- **주요 불편 사항 1 (중복/오인)**: 예약 현황표와 점검 이력이 별개 문서로 관리되어, 점검 중인 시설을 빈 시설로 오인해 예약하려는 행정 착오 발생.
- **주요 불편 사항 2 (이력 단절)**: 설비 고장 수리 이력과 반복 결함 여부를 파악하기 어려워 행사 직전 장비 고장으로 훈련 차질 발생.
- **AI 도우미 도입 시 기대 효과**: 조건별 빈 시설 10초 내 즉시 탐색, 점검 이력 일괄 요약, 미보유 데이터 자동 식별 및 담당 부서 유선 안내로 행정 확인 소요 80% 이상 단축.

## 4. 심화 도전 과제: 자가 경험 기반 추가 질문 (질문 21번)
- **추가 질문 내용**: 금요일 야간(19-21)에 영화 상영 외에 대대 강당 개별 이용이 가능한가요?
- **분류 및 검증 결과**: 분류 - 예약 가능 여부 / 자료에 답이 있는가 - O (금요일 19-21시는 '영화 상영'으로 지정되어 있어 개별 이용 불가 판정)


### 🔍 평가 기준 (PASS 조건)
<h4>미션 판정</h4><p><strong>평가 구분:</strong> 필수</p><p><strong>핵심 평가 기준:</strong> [필수] 문제 정의·AI 적용 판단 — 시설 정보 검색의 불편과 병목 요소를 도출하고 AI 검색 도우미가 필요한 이유를 타당하게 제시했는가? (PASS / FAIL)</p><h4>미션 PASS 조건</h4><ul><li><p>질문 카드 20개 유형 분류표 완성 및 자료 밖/대행 불가 질문(16~20번) 식별</p></li><li><p>기존 수작업 확인 방식 소요 시간 측정 및 실무 병목 요인 분석서 작성</p></li><li><p>교육용 가상 데이터 활용 및 군사보안 수칙 엄수</p></li></ul><h4>채점자 판정 메모</h4><blockquote><p>단순한 표 작성을 넘어, 자료 밖 질문(16~19번)과 예약 대행 요청(20번)을 AI가 함부로 답해서는 안 되는 '인간 책임/예외 영역'으로 명확히 인식했는지 확인한다.</p></blockquote>


In [ ]:
# [미션 01] 프롬프트 및 실습 결과 기록
mission_01_prompt = """# [미션 01] 질문 카드 20개 유형 분류 및 행정 병목 분석 질의
너는 군 시설 관리 행정 분석관이다.
가상 한빛대대 장병들이 시설 이용과 관련해 자주 묻는 질문 카드 20개를 아래 4가지 유형으로 분류하고,
기존 수작업 확인 방식(게시판 표 확인, 유선 내선 문의)의 병목 요인과 소요 시간을 분석해 줘.

[질문 카드 20개]
1. 수요일 저녁에 빈 강당 있어요?
2. 목요일 오후에 세미나실 A, B 중 빈 곳은?
3. 금요일 오전에 강당 쓸 수 있나요?
4. 토요일에 하루 종일 비는 시설이 있나요?
5. 월요일 오후 세미나실 A와 B가 둘 다 비는 시간은?
6. 2중대가 이번 주 강당을 쓰는 시간은?
7. 수요일에 체력단련실을 단체로 쓸 수 있는 시간은?
8. 강당의 점검·고장 이력을 정리해 줘
9. 세미나실 B 빔프로젝터는 몇 번 고장 났어요?
10. 가장 고장 신고가 잦은 시설은 어디예요?
11. 아직 수리가 끝나지 않은 건은?
12. 고장 신고 처리에 평균 며칠 걸렸어요?
13. 강당 무선 마이크 고장 날짜를 알려 줘
14. 정기점검을 가장 오래 안 한 시설은?
15. 반복해서 고장 나는 설비 목록을 알려 줘
16. 세미나실 C는 비어 있어요?
17. 강당 최대 수용 인원이 몇 명이에요?
18. 다음 주 월요일 강당 예약 현황은?
19. 강당 예약 승인해 주는 담당자 이름은?
20. 빈 강당 있으면 지금 예약해 줘

[분류 기준]
1. 예약 가능 여부: 특정 요일/시간대 시설 빈자리 확인 (자료 내 답변 가능)
2. 점검·고장 이력: 설비 점검 현황, 소요일, 고장 내역 확인 (자료 내 답변 가능)
3. 자료 밖 질문: 부대 주간 예약표 및 과거 이력 범위 밖의 질문 (자료 없음)
4. 대행 불가 요청: 예약 승인·대행 등 AI의 권한을 넘어서는 요청 (행정 절차상 거부)
"""

mission_01_result = """### 📋 질문 카드 20개 유형 분류 및 실무 분석 보고서

#### 1. 질문 카드 20개 분류 집계표
| 카드 번호 | 질문 내용 | 분류 | 자료 내 답변 가능 여부 |
| :---: | :--- | :---: | :---: |
| 1 | 수요일 저녁에 빈 강당 있어요? | 예약 가능 여부 | O |
| 2 | 목요일 오후에 세미나실 A, B 중 빈 곳은? | 예약 가능 여부 | O |
| 3 | 금요일 오전에 강당 쓸 수 있나요? | 예약 가능 여부 | O |
| 4 | 토요일에 하루 종일 비는 시설이 있나요? | 예약 가능 여부 | O |
| 5 | 월요일 오후 세미나실 A와 B가 둘 다 비는 시간은? | 예약 가능 여부 | O |
| 6 | 2중대가 이번 주 강당을 쓰는 시간은? | 예약 가능 여부 | O |
| 7 | 수요일에 체력단련실을 단체로 쓸 수 있는 시간은? | 예약 가능 여부 | O |
| 8 | 강당의 점검·고장 이력을 정리해 줘 | 점검·고장 이력 | O |
| 9 | 세미나실 B 빔프로젝터는 몇 번 고장 났어요? | 점검·고장 이력 | O |
| 10 | 가장 고장 신고가 잦은 시설은 어디예요? | 점검·고장 이력 | O |
| 11 | 아직 수리가 끝나지 않은 건은? | 점검·고장 이력 | O |
| 12 | 고장 신고 처리에 평균 며칠 걸렸어요? | 점검·고장 이력 | O |
| 13 | 강당 무선 마이크 고장 날짜를 알려 줘 | 점검·고장 이력 | O |
| 14 | 정기점검을 가장 오래 안 한 시설은? | 점검·고장 이력 | O |
| 15 | 반복해서 고장 나는 설비 목록을 알려 줘 | 점검·고장 이력 | O |
| 16 | 세미나실 C는 비어 있어요? | 자료 밖 질문 | X (존재하지 않는 시설) |
| 17 | 강당 최대 수용 인원이 몇 명이에요? | 자료 밖 질문 | X (수용 인원 미기재) |
| 18 | 다음 주 월요일 강당 예약 현황은? | 자료 밖 질문 | X (금주 데이터만 보유) |
| 19 | 강당 예약 승인해 주는 담당자 이름은? | 자료 밖 질문 | X (인적 정보 미보유) |
| 20 | 빈 강당 있으면 지금 예약해 줘 | 대행 불가 요청 | X (행정 권한 밖 요청) |

- **집계 요약**: 예약 가능 여부 7건, 점검·고장 이력 8건, 자료 밖 질문 4건, 대행 불가 1건 (총 20건)

#### 2. 현재 검색 방식 및 소요 시간 분석표
| 확인 방식 | 수행 절차 | 건당 평균 소요 시간 | 주요 지연 요인 |
| :--- | :--- | :---: | :--- |
| 게시판 종이 표 확인 | 행정반 이동 -> 게시판 열람 -> 대조 | 약 15분 | 표 갱신 지연, 이동 시간 소요, 수작업 대조 착오 위험 |
| 담당자 내선 문의 | 행정과/군수과 전화 -> 확인 요청 -> 대기 | 약 20분 | 담당관 부재(훈련, 순찰, 회의) 시 지연, 반복 문의로 인한 행정 피로 |

#### 3. 현장 불편 요소 및 AI 검색 도우미 도입 필요성
- **주요 불편 사항 1 (중복/오인)**: 예약표와 점검 이력이 별개 문서로 관리되어, 점검 중인 시설을 빈 시설로 오인해 예약하려는 행정 착오 발생.
- **주요 불편 사항 2 (이력 단절)**: 설비 고장 수리 이력과 반복 결함 여부를 파악하기 어려워 행사 직전 장비 고장으로 훈련 차질 발생.
- **AI 도우미 도입 시 기대 효과**: 조건별 빈 시설 10초 내 즉시 탐색, 점검 이력 일괄 요약, 미보유 데이터 자동 식별 및 담당 부서 유선 안내로 행정 확인 소요 80% 이상 단축.
"""

print(f"미션 01 실습 기록 완료: 프롬프트 {len(mission_01_prompt.strip())}자, 결과 {len(mission_01_result.strip())}자")


In [ ]:
# [데이터 교차 검산 - 미션1] 질문 카드 20개 유형 분류 집계 스크립트
# 질문 카드 20개 분류가 올바르게 이루어졌는지 검산합니다.
questions_data = [
    {"id": 1, "type": "예약 가능 여부"}, {"id": 2, "type": "예약 가능 여부"},
    {"id": 3, "type": "예약 가능 여부"}, {"id": 4, "type": "예약 가능 여부"},
    {"id": 5, "type": "예약 가능 여부"}, {"id": 6, "type": "예약 가능 여부"},
    {"id": 7, "type": "예약 가능 여부"},
    {"id": 8, "type": "점검·고장 이력"}, {"id": 9, "type": "점검·고장 이력"},
    {"id": 10, "type": "점검·고장 이력"}, {"id": 11, "type": "점검·고장 이력"},
    {"id": 12, "type": "점검·고장 이력"}, {"id": 13, "type": "점검·고장 이력"},
    {"id": 14, "type": "점검·고장 이력"}, {"id": 15, "type": "점검·고장 이력"},
    {"id": 16, "type": "자료 밖 질문"}, {"id": 17, "type": "자료 밖 질문"},
    {"id": 18, "type": "자료 밖 질문"}, {"id": 19, "type": "자료 밖 질문"},
    {"id": 20, "type": "대행 불가 요청"}
]
df_q = pd.DataFrame(questions_data)
print("=== 질문 카드 유형별 건수 집계 ===")
print(df_q['type'].value_counts())
print("\n총 질문 수:", len(df_q), "건 (자료 밖 및 대행 불가: 5건 확인)")
